# Master Supervised Benchmark Summary: Cyber Attack Detection in UAVs

## 1. Executive Summary
This notebook aggregates and analyzes the results across **all supervised model experiments** conducted in the `experiments/` workspace for the **UAV Cyber Attack Detection** project.

### Models Evaluated:
- **Physical Domain:** Random Forest (Default & Balanced), Extra Trees, Linear SVM, RBF SVM (Tuned Balanced), MLP Neural Network.
- **Cyber Domain (Full 5 Classes):** Random Forest (Default & Low FAR), Extra Trees, Linear SVM (Default & Balanced), Nystroem RBF Kernel SVM, MLP Neural Network.

### Standard 5-Class Canonical Taxonomy:
1. `Benign` (Normal operational state)
2. `DoS` (Denial of Service packet flood)
3. `Replay` (Replayed historical telemetry commands)
4. `Evil_Twin` (Spoofed Ground Control Station)
5. `FDI` (False Data Injection targeting IMU/position estimates)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, Markdown

sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 11

## 2. Benchmark Metrics Table
Comprehensive metrics tracking Detection Accuracy, Macro-F1, False Alarm Rate (FAR on Benign), Inference Latency, and Model Storage Size.

In [ ]:
results_csv = "results/master_metrics_summary.csv"
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    display(df[[
        "Domain", "Model", "Accuracy (%)", "Macro F1 (%)", 
        "False Alarm Rate (%)", "Latency (us/sample)", "Model Size (KB)",
        "F1: Benign (%)", "F1: DoS (%)", "F1: Replay (%)", "F1: Evil_Twin (%)", "F1: FDI (%)"
    ]])
else:
    print("[!] Run 'generate_master_report.py' first to populate results.")

## 3. Comparative Visualizations
Performance comparison across domains and models.

In [ ]:
chart_file = "results/model_comparison_plots.png"
if os.path.exists(chart_file):
    display(Image(filename=chart_file))
else:
    print("Chart file not found.")

## 4. UAV Edge Deployment: Pareto Frontier (Latency vs. F1-Score)
Evaluating the optimal trade-off between attack detection capability ($F_1$-score) and onboard computational efficiency (microseconds per sample).

In [ ]:
pareto_file = "results/latency_vs_f1_pareto.png"
if os.path.exists(pareto_file):
    display(Image(filename=pareto_file))
else:
    print("Pareto plot not found.")

## 5. Key Research Takeaways

### 1. The Domain Complementarity Principle
- **Physical Telemetry** is almost infallible at detecting **kinematic-shifting attacks** (`Evil_Twin` and `FDI` have $>99.7\%$ F1) because they cause violent angular/position deviations, but struggles to identify network packet floods (`DoS` $F_1 \approx 47\%$, `Replay` $F_1 \approx 60\%$).
- **Cyber Network Traffic** easily captures **packet rate and frame anomalies** (`DoS` reaches $>70\%$ F1 in Extra Trees and FAR drops to $2.55\%$ in ensembles).

### 2. Edge Hardware Efficiency
- **Linear SVM:** Fast inference ($0.2\text{--}0.3\;\mu\text{s/sample}$), suitable for high-speed hardware line-rate filtering.
- **Decision Tree Ensembles (RF / ExtraTrees):** Best balance of detection power ($78\text{--}83\%$ Macro F1) and manageable latency ($2\text{--}8\;\mu\text{s/sample}$).
- **Kernel SVM (RBF):** High inference latency ($50\text{--}160\;\mu\text{s/sample}$), limiting its suitability for edge microcontrollers.

### 3. Next Research Steps
- **Step 1:** Implement Gradient Boosted Decision Trees (**LightGBM & XGBoost**) to optimize tabular performance.
- **Step 2:** Formulate **Cyber-Physical Multimodal Fusion** (Early and Late) to close the Replay/DoS gap.
- **Step 3:** Use these upper-bound supervised metrics to benchmark **Unsupervised Anomaly Detection** (Autoencoders, Isolation Forests, One-Class SVM).